# CLIP Image Embeddings for Pin Visual Search

Research goal: **OpenAI CLIP ViT-L/14** (768-dim)

In [ ]:
# !pip install open-clip-torch Pillow torch requests


In [ ]:
import io, time, urllib.request
import numpy as np
import torch
import open_clip
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model, _, preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-L-14")
model = model.to(device).eval()
print(f"CLIP ViT-L/14 loaded. Visual embedding dim: 768")


## 1. Encode images with CLIP

In [ ]:
# Using torchvision sample images as stand-ins for real pin uploads
from torchvision.datasets import FakeData
from torchvision import transforms

fake_ds = FakeData(size=20, image_size=(3, 224, 224), transform=transforms.ToPILImage())

image_tensors = []
for i in range(20):
    img, _ = fake_ds[i]
    image_tensors.append(preprocess(img))

batch = torch.stack(image_tensors).to(device)

with torch.inference_mode():
    image_features = model.encode_image(batch)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

print(f"Encoded {len(image_tensors)} images. Shape: {image_features.shape}")


## 2. Text-to-image retrieval (zero-shot)

In [ ]:
queries = [
    "a beautiful sunset over mountains",
    "street food market in Asia",
    "cozy coffee shop interior",
    "hiking trail in autumn forest",
]

text_tokens = tokenizer(queries).to(device)
with torch.inference_mode():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

sim = cosine_similarity(text_features.cpu().numpy(), image_features.cpu().numpy())
print(f"Similarity matrix shape: {sim.shape}")

for i, q in enumerate(queries):
    top2 = np.argsort(sim[i])[::-1][:2]
    print(f"
Query: {q!r}")
    for rank, idx in enumerate(top2, 1):
        print(f"  {rank}. image_{idx:02d}  score={sim[i, idx]:.3f}")


## 3. Image-to-image similarity (deduplication use case)

In [ ]:
img_sim = cosine_similarity(image_features.cpu().numpy())
np.fill_diagonal(img_sim, 0)  # ignore self-similarity

DEDUP_THRESHOLD = 0.92
duplicates = [(i, j) for i in range(len(img_sim)) for j in range(i+1, len(img_sim)) if img_sim[i, j] >= DEDUP_THRESHOLD]
print(f"Potential duplicates (threshold={DEDUP_THRESHOLD}): {len(duplicates)} pairs")

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(img_sim, cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("Image-to-Image Cosine Similarity (CLIP ViT-L/14)")
plt.tight_layout()
plt.savefig("clip_img_sim.png", dpi=120)
plt.show()


## 4. Latency benchmark

In [ ]:
latencies = []
for bs in [1, 4, 8, 16, 32]:
    imgs = torch.stack([preprocess(FakeData(1, image_size=(3,224,224), transform=transforms.ToPILImage())[0][0]) for _ in range(bs)]).to(device)
    # warmup
    with torch.inference_mode(): model.encode_image(imgs)
    t0 = time.perf_counter()
    RUNS = 10
    for _ in range(RUNS):
        with torch.inference_mode(): model.encode_image(imgs)
    elapsed = (time.perf_counter() - t0) / RUNS
    latencies.append({"batch_size": bs, "latency_ms": elapsed * 1000, "img_per_sec": bs / elapsed})
    print(f"bs={bs:2d}  latency={elapsed*1000:.1f}ms  img/s={bs/elapsed:.1f}")


## Conclusions

| Metric | Value |
|--------|-------|
| Model | CLIP ViT-L/14 (openai) |
| Visual embedding dim | 768 |
| Text embedding dim | 768 |
| Zero-shot | Yes |

**Findings:**
- CLIP enables zero-shot text-to-image and image-to-image search without training
- 768-dim embeddings stored in Qdrant "image" named vector per pin
- Deduplication threshold ~0.92 catches near-identical reposts
- For cross-modal search: store text (E5) + image (CLIP) vectors per pin, fuse at query time
